Bygger en CSV fil med server objekt, att populera ett datacenter med.

In [33]:
import csv

In [34]:
csv_data = """Web-Server-01,192.168.1.10,16
Web-Server-02,192.168.1.11,16
Web-Server-03,192.168.1.12,8
App-Server-01,192.168.1.13,32
App-Server-02,192.168.1.14,32
App-Server-03,192.168.1.15,16
DB-Master-01,192.168.1.16,64
DB-Slave-01,192.168.1.17,32
DB-Slave-02,192.168.1.18,32
Backup-Server,192.168.1.19,8
Proxy-01,192.168.1.20,4
Proxy-02,192.168.1.21,4
Auth-Server,192.168.1.22,8
DNS-Server-01,192.168.1.23,4
DNS-Server-02,192.168.1.24,4
Monitoring-01,192.168.1.25,16
Storage-01,192.168.1.26,12
Storage-02,192.168.1.27,12
Dev-Server-01,192.168.1.28,8
Test-Server-01,192.168.1.29,4"""

with open("server.csv", "w", encoding="utf-8") as file:
    file.write(csv_data.strip())

print("En seed file för att populera datacenter med 20 enheter har skapats")

En seed file för att populera datacenter med 20 enheter har skapats


Ett första bygge av Klasser med arv

In [35]:
# Parent/Child klasser

class Hardware:
    def __init__(self, name: str, ip_address: str):
        self.name = name
        self.ip_address = ip_address
        self.status = "OK" 
        
class Server(Hardware):
    def __init__(self, name: str, ip_address: str, cpu_cores: int):
        super().__init__(name, ip_address)
        self.cpu_cores = cpu_cores

print("Parent Klass Hardware och Child  Klass Server har nu definierats i en första version")

Parent Klass Hardware och Child  Klass Server har nu definierats i en första version


Nu ska vi läsa in csv filen i internminnet i en så kallad lista (inventory[]), för att därefter populera upp datacenter

# inläsning av CSV fil

In [36]:
inventory = []

with open("server.csv", "r", encoding ="utf-8") as file:
    reader = csv.reader(file)
    for row in reader:
        server_object = Server(name=row[0], ip_address=row[1], cpu_cores=int(row[2]))
        inventory.append(server_object)
        
print(f"totalt antal inlästa server objekt är {len(inventory)}")
 

totalt antal inlästa server objekt är 20


Nu ska vi testa att läsa från inventory listan i minnet

In [37]:
print("läser vad som finns på rad två i listan (index startar från 0)")
print(f"namnet är:{inventory[1].name} IP är:{inventory[1].ip_address} antal cpu är {inventory[1].cpu_cores}")
print("\nläser innehållet i de tre första raderna i listan")
for srv in inventory[:3]:
    print(f"namnet är: {srv.name} IP är: {srv.ip_address} antal cpu är: {srv.cpu_cores}")        



läser vad som finns på rad två i listan (index startar från 0)
namnet är:Web-Server-02 IP är:192.168.1.11 antal cpu är 16

läser innehållet i de tre första raderna i listan
namnet är: Web-Server-01 IP är: 192.168.1.10 antal cpu är: 16
namnet är: Web-Server-02 IP är: 192.168.1.11 antal cpu är: 16
namnet är: Web-Server-03 IP är: 192.168.1.12 antal cpu är: 8


Nu testas förändring av värden i listan

In [38]:
print(f"vi ändrar antal cpu i Web-Server-3 från {inventory[2].cpu_cores} till {2*inventory[2].cpu_cores}")

# --- 4. FÖRÄNDRA ETT OBJEKT I LISTAN ---
# Vi hämtar databasservern (index 6 i listan) och simulera ett fel
ws_3 = inventory[2]
ws_3.cpu_cores = 2*inventory[2].cpu_cores 

print(f"{ws_3.name} har nu {ws_3.cpu_cores} cpu kärnor ")


for rader in inventory[:3]:
    rader.status = "Warning"
    print (rader.status, rader.name, rader.cpu_cores)   

for rader in inventory[:]:
    if rader.status == "Warning":
        rader.status = "CRITICAL"
        print(rader.status, rader.name, rader.cpu_cores)  
    
 

vi ändrar antal cpu i Web-Server-3 från 8 till 16
Web-Server-03 har nu 16 cpu kärnor 
Warning Web-Server-01 16
Warning Web-Server-02 16
Warning Web-Server-03 16
CRITICAL Web-Server-01 16
CRITICAL Web-Server-02 16
CRITICAL Web-Server-03 16


Nu ska vi ID tagga alla server objekt 
Målet är att varje server skall få en ID tagg (IT-01, IT-02 osv) 

In [39]:
id_counter = 1

class Hardware:
    def __init__(self, name: str, ip_address: str):
        global id_counter
        # varje NY instans får ett unikt ID när __init__ körs
        self.id_tag = f"ID-{id_counter}"
        id_counter += 1
        
        self.name = name
        self.ip_address = ip_address
        self.status = "OK"
        
        
class Server(Hardware):
    def __init__(self, name: str, ip_address: str, cpu_cores: int):
        # super() anropar Hardware.__init__ som sätter id_tag, name, ip_adress och status
        super().__init__(name, ip_address)
        self.cpu_cores = cpu_cores
        

inventory = []

with open("server.csv", "r", encoding="utf-8") as file:
    reader = csv.reader(file)
    
    for row in reader:
        # När server() anropas körs __init__ så att id_tag skapas automatiskt
        server_object = Server(name=row[0], ip_address=row[1], cpu_cores=int(row[2]))
        inventory.append(server_object)
    

for srv in inventory:
    print(f"{srv.id_tag} {srv.name}")
    
print("Nu har alla servrar id taggats via en ny attribut och metod i Parent Class hardware")

ID-1 Web-Server-01
ID-2 Web-Server-02
ID-3 Web-Server-03
ID-4 App-Server-01
ID-5 App-Server-02
ID-6 App-Server-03
ID-7 DB-Master-01
ID-8 DB-Slave-01
ID-9 DB-Slave-02
ID-10 Backup-Server
ID-11 Proxy-01
ID-12 Proxy-02
ID-13 Auth-Server
ID-14 DNS-Server-01
ID-15 DNS-Server-02
ID-16 Monitoring-01
ID-17 Storage-01
ID-18 Storage-02
ID-19 Dev-Server-01
ID-20 Test-Server-01
Nu har alla servrar id taggats via en ny attribut och metod i Parent Class hardware


Nu ska vi göra samma sak men genom att arbeta med listan i minnet och 
utöka med en label som heter id_tag, och inte bara instansera om alla objekt 

In [40]:
inventory = []
with open("server.csv", "r", encoding="utf-8") as file:
    reader = csv.reader(file)
    for row in reader:    
        server_object = Server(name=row[0], ip_address=row[1], cpu_cores=int(row[2]))
        inventory.append(server_object)

print("Testar att skriva ut rad 3 för att verifiera att en lista skapats i minnet")
print (inventory[2].name)


Testar att skriva ut rad 3 för att verifiera att en lista skapats i minnet
Web-Server-03


Nu ska vi lägga till en klassvariabel i listan med de 
instanserade servrarna som redan ligger i minnet
Det gör vi genom att sätta en klassvariabel DIREKT 
på Parent Class Hardware genom hardware.datacenter_site = "DC_STOCKHOLM_ZONE_1"
Nu finns denna egenskap, dvs label sitenamn för varje server, AUTOMATISKT tillagd i samtliga serverobjekt i minnet 

In [41]:
print("Vi sätter den nya klassvariabeln datacenter_site till DC_STOCKHOLM_ZONE_1")
Hardware.datacenter_site = "DC_STOCKHOLM_ZONE_1"

print("Testar att skriva ut rad 3 för att verifiera att klassvariabeln skapats i minnet")
print (inventory[2].datacenter_site)

Vi sätter den nya klassvariabeln datacenter_site till DC_STOCKHOLM_ZONE_1
Testar att skriva ut rad 3 för att verifiera att klassvariabeln skapats i minnet
DC_STOCKHOLM_ZONE_1


Nu ska vi lägga till id_tagg till listan med de 
instanserade servrarna som redan ligger i minnet
Detta genom en for loop som säkerhetstaggar 
BEFINTLIG population, alltså jag instasierar INTE en ny.
Vi stegar igenom listan i minnet och sätter en ny instansvariabel (.security_tag)

In [ ]:

for server in inventory:
    if server.cpu_cores >=32:
        security_tag = "High_Risk"
    elif server.cpu_cores >=16:
        security_tag = "Medium_Risk"
    else:
        server.security_tag = "Low_Risk"

for srv in inventory[:8]:
    print(f"{srv.name}, {srv.cpu_cores}, {srv.security_tag}")
    


AttributeError: 'Server' object has no attribute 'security_tag'